# Error Handling & Retry（超时 / 限流 / 网络错误）

## 目标

本示例演示如何区分超时、429 限流、网络连接错误和 5xx 服务端错误，并通过带随机抖动的指数退避进行有限次数重试。

## 前置条件

- Python 3.10+
- OpenAI Python SDK `openai>=1.0.0`，环境变量加载库 `python-dotenv>=1.0.0`
- 必需环境变量：`HY3_BASE_URL`、`HY3_API_KEY`、`HY3_MODEL`
- 可选环境变量：`HY3_TIMEOUT_SECONDS=30`、`HY3_MAX_RETRIES=3`、`HY3_RETRY_BASE_SECONDS=1`、`HY3_RETRY_MAX_SECONDS=30`
- 模型能力要求：仅需支持非流式 Chat Completions；错误类型主要由 SDK、网络、服务端和网关决定

安装依赖：

```bash
python -m pip install "openai>=1.0.0" "python-dotenv>=1.0.0"
```

## 完整请求

示例把 SDK 内置重试设为 `0`，以便清楚展示调用方自己的重试与退避逻辑。生产代码应避免同时叠加多层未知次数的重试。

```python
client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    timeout=TIMEOUT_SECONDS,
    max_retries=0,
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "请用一句话解释什么是指数退避。"}
    ],
    temperature=0.9,
    top_p=1.0,
    max_tokens=128,
    extra_body={
        "chat_template_kwargs": {"reasoning_effort": "no_think"}
    },
)
```

## 完整 Response 解析

脚本捕获并分类以下异常：

- `APITimeoutError`：连接、写入、读取或连接池等待超时。
- `RateLimitError`：服务端或网关返回 HTTP 429；优先读取数字形式的 `Retry-After`。
- `APIConnectionError`：DNS、连接拒绝、连接重置等网络错误。
- `APIStatusError`：仅对 HTTP 5xx 重试；400、401、403、404 等非暂时性错误直接抛出。

```python
for attempt in range(MAX_RETRIES + 1):
    try:
        response = client.chat.completions.create(...)
        break
    except (RateLimitError, APITimeoutError, APIConnectionError, APIStatusError) as exc:
        category = retry_category(exc)
        if category is None or attempt >= MAX_RETRIES:
            raise
        delay = compute_delay(attempt, exc)
        time.sleep(delay)
```

成功后会解析响应 ID、模型、全部 choices、正文、结束原因和 Token 用量。退避公式为 `base * 2^retry_index + jitter`，并受 `HY3_RETRY_MAX_SECONDS` 限制；429 的数字 `Retry-After` 优先。

## 运行方式

在 `quickstart/` 目录执行：

```bash
python examples/06_error_handling/error_handling.py
```

可将 `HY3_BASE_URL` 临时设为不可访问地址以观察网络错误；不要在生产服务上故意制造高并发来测试 429。

## 示例输出

```text
第 1 次请求失败（网络错误），1.08s 后进行第 1 次重试
第 2 次请求失败（限流），2.00s 后进行第 2 次重试

=== 请求成功 ===
id=chatcmpl-retry-example
model=hy3
choice[0].finish_reason=stop
choice[0].content=指数退避是在连续失败后逐步延长重试等待时间的策略。
usage: prompt=19, completion=24, total=43
```

实际错误顺序、等待时间、ID 和 Token 数会随环境变化。


In [ ]:
"""Hy3 API 超时、限流、网络错误与服务端错误的重试示例。"""

import os
import random
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import (
    APIConnectionError,
    APIStatusError,
    APITimeoutError,
    OpenAI,
    RateLimitError,
)


def load_project_env() -> None:
    candidates = [Path.cwd() / ".env", Path.cwd() / "quickstart" / ".env"]
    if "__file__" in globals():
        candidates.insert(0, Path(__file__).resolve().parents[2] / ".env")
    for candidate in candidates:
        if candidate.is_file():
            load_dotenv(candidate, override=False)
            return


load_project_env()

BASE_URL = os.getenv("HY3_BASE_URL", "http://127.0.0.1:8000/v1")
API_KEY = os.getenv("HY3_API_KEY", "EMPTY")
MODEL = os.getenv("HY3_MODEL", "hy3")
TIMEOUT_SECONDS = float(os.getenv("HY3_TIMEOUT_SECONDS", "30"))
MAX_RETRIES = int(os.getenv("HY3_MAX_RETRIES", "3"))
RETRY_BASE_SECONDS = float(os.getenv("HY3_RETRY_BASE_SECONDS", "1"))
RETRY_MAX_SECONDS = float(os.getenv("HY3_RETRY_MAX_SECONDS", "30"))


def retry_after_seconds(exc) -> float | None:
    """读取限流响应中的 Retry-After 秒数；非数字格式交给指数退避。"""
    response = getattr(exc, "response", None)
    if response is None:
        return None
    value = response.headers.get("retry-after")
    if value is None:
        return None
    try:
        return max(0.0, float(value))
    except ValueError:
        return None


def retry_category(exc) -> str | None:
    if isinstance(exc, RateLimitError):
        return "限流"
    if isinstance(exc, APITimeoutError):
        return "超时"
    if isinstance(exc, APIConnectionError):
        return "网络错误"
    if isinstance(exc, APIStatusError) and exc.status_code >= 500:
        return f"服务端错误 HTTP {exc.status_code}"
    return None


def compute_delay(retry_index: int, exc) -> float:
    retry_after = retry_after_seconds(exc)
    if retry_after is not None:
        return min(RETRY_MAX_SECONDS, retry_after)
    exponential = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2**retry_index))
    jitter = random.uniform(0.0, min(1.0, exponential * 0.1))
    return min(RETRY_MAX_SECONDS, exponential + jitter)


def create_completion_with_retry(client: OpenAI):
    """最多执行 1 + MAX_RETRIES 次请求，并只重试暂时性错误。"""
    for attempt in range(MAX_RETRIES + 1):
        try:
            return client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": "请用一句话解释什么是指数退避。",
                    }
                ],
                temperature=0.9,
                top_p=1.0,
                max_tokens=128,
                extra_body={
                    "chat_template_kwargs": {
                        "reasoning_effort": "no_think",
                    }
                },
            )
        except (RateLimitError, APITimeoutError, APIConnectionError, APIStatusError) as exc:
            category = retry_category(exc)
            if category is None:
                # 400/401/403/404 等通常不是短暂故障，立即交给调用方处理。
                raise
            if attempt >= MAX_RETRIES:
                print(f"{category}：已用尽 {MAX_RETRIES} 次重试")
                raise

            delay = compute_delay(attempt, exc)
            print(
                f"第 {attempt + 1} 次请求失败（{category}），"
                f"{delay:.2f}s 后进行第 {attempt + 1} 次重试"
            )
            time.sleep(delay)

    raise RuntimeError("不可达分支")


def parse_response(response) -> None:
    print("\n=== 请求成功 ===")
    print(f"id={response.id}")
    print(f"model={response.model}")
    for choice in response.choices:
        print(f"choice[{choice.index}].finish_reason={choice.finish_reason}")
        print(f"choice[{choice.index}].content={choice.message.content}")
    if response.usage:
        print(
            "usage: "
            f"prompt={response.usage.prompt_tokens}, "
            f"completion={response.usage.completion_tokens}, "
            f"total={response.usage.total_tokens}"
        )


def main() -> None:
    # 关闭 SDK 内置重试，确保示例中的退避过程清晰可见。
    client = OpenAI(
        base_url=BASE_URL,
        api_key=API_KEY,
        timeout=TIMEOUT_SECONDS,
        max_retries=0,
    )
    response = create_completion_with_retry(client)
    parse_response(response)


if __name__ == "__main__":
    main()
